<div dir="rtl">

# الدرس 52: تشغيل نموذج جاهز على جهازك

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

### نموذجك من الدرس 49

</div>

In [ ]:
import math
from dataclasses import dataclass
import torch
import torch.nn.functional as F
from torch import nn

@dataclass
class GPTConfig:
    vocab_size: int = 50257
    block_size: int = 1024          # context window
    n_layer: int = 12
    n_head: int = 12
    d_model: int = 768
    dropout: float = 0.0

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.dropout = cfg.dropout

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.dropout if self.training else 0.0)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc = nn.Linear(cfg.d_model, 4 * cfg.d_model)
        self.proj = nn.Linear(4 * cfg.d_model, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x))))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.mlp = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))      # tokens communicate
        x = x + self.mlp(self.ln2(x))       # each token computes
        return x

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight            # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, m):                            # GPT-2 style initialization
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size, "sequence longer than the context window"
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        logits = self.head(self.ln_f(x))                   # (B, T, vocab_size)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

In [ ]:
import os
if not os.path.exists("tiny_gpt.pt"):
    raise FileNotFoundError("Run the notebook of lesson 49 first: it trains and saves tiny_gpt.pt")
ckpt = torch.load("tiny_gpt.pt")
cfg = GPTConfig(**ckpt["config"])
ours = GPT(cfg)
ours.load_state_dict(ckpt["model"])
ours.eval()
chars = ckpt["chars"]
print("our model:", sum(p.numel() for p in ours.parameters()), "parameters")

<div dir="rtl">

### النقل إلى جي بي تي 2

</div>

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel

hf_cfg = GPT2Config(vocab_size=cfg.vocab_size, n_positions=cfg.block_size, n_embd=cfg.d_model,
                    n_layer=cfg.n_layer, n_head=cfg.n_head, activation_function="gelu",
                    resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0,
                    bos_token_id=None, eos_token_id=None)
hf_model = GPT2LMHeadModel(hf_cfg)

sd = ours.state_dict()
new = {"transformer.wte.weight": sd["tok_emb.weight"],
       "transformer.wpe.weight": sd["pos_emb.weight"],
       "transformer.ln_f.weight": sd["ln_f.weight"], "transformer.ln_f.bias": sd["ln_f.bias"]}
names = {"ln1": "ln_1", "ln2": "ln_2", "attn.qkv": "attn.c_attn", "attn.proj": "attn.c_proj",
         "mlp.fc": "mlp.c_fc", "mlp.proj": "mlp.c_proj"}
for i in range(cfg.n_layer):
    for ours_name, hf_name in names.items():
        w, b = sd[f"blocks.{i}.{ours_name}.weight"], sd[f"blocks.{i}.{ours_name}.bias"]
        if not ours_name.startswith("ln"):
            w = w.T                                    # GPT-2 stores linear weights transposed
        new[f"transformer.h.{i}.{hf_name}.weight"] = w
        new[f"transformer.h.{i}.{hf_name}.bias"] = b
missing, unexpected = hf_model.load_state_dict(new, strict=False)
print("missing:", missing, "| unexpected:", unexpected)

hf_model.eval()
x = torch.tensor([[chars.index(c) for c in "ROMEO:\nWhat light"]])
with torch.no_grad():
    diff = (ours(x)[0] - hf_model(x).logits).abs().max().item()
print("largest difference between the two models:", diff)

<div dir="rtl">

### مقطّع حروف بصيغة المكتبة

</div>

In [ ]:
from tokenizers import Regex, Tokenizer, decoders, models, pre_tokenizers
from transformers import PreTrainedTokenizerFast

tk = Tokenizer(models.WordLevel(vocab={c: i for i, c in enumerate(chars)}, unk_token=" "))
tk.pre_tokenizer = pre_tokenizers.Split(Regex(r"[\s\S]"), behavior="isolated")   # every character
tk.decoder = decoders.Fuse()                                                     # join without spaces
tokenizer = PreTrainedTokenizerFast(tokenizer_object=tk)

hf_model.save_pretrained("my-tiny-gpt")
tokenizer.save_pretrained("my-tiny-gpt")
print(sorted(os.listdir("my-tiny-gpt")))

<div dir="rtl">

### التحميل بالأصناف التلقائية

</div>

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, set_seed

tok = AutoTokenizer.from_pretrained("my-tiny-gpt")
model = AutoModelForCausalLM.from_pretrained("my-tiny-gpt")
print(type(model).__name__)

inputs = tok("JULIET:\n", return_tensors="pt")
set_seed(0)
room = model.config.n_positions - inputs["input_ids"].shape[1]      # the whole text must fit in 64 positions
out = model.generate(**inputs, max_new_tokens=room, do_sample=True, temperature=0.8, top_p=0.9, pad_token_id=0)
print(tok.decode(out[0]))

<div dir="rtl">

### خط الأنابيب

</div>

In [ ]:
generator = pipeline("text-generation", model="my-tiny-gpt")
set_seed(1)
result = generator("KING RICHARD III:\n", max_new_tokens=46, do_sample=True, temperature=0.8, pad_token_id=0)
print(result[0]["generated_text"])

<div dir="rtl">

### حاسبة الذاكرة

</div>

In [ ]:
def weights_gb(n_params, bits=16):
    return n_params * bits / 8 / 1e9

def kv_cache_gb(n_tokens, n_layers, n_kv_heads, head_dim, bits=16):
    return 2 * n_layers * n_kv_heads * head_dim * n_tokens * bits / 8 / 1e9

for name, n, layers, kv_heads, head_dim in [("hypothetical 0.5B", 0.5e9, 24, 2, 64),
                                            ("hypothetical 1.5B", 1.5e9, 28, 2, 128),
                                            ("hypothetical 7B", 7e9, 28, 4, 128)]:
    cache = kv_cache_gb(8192, layers, kv_heads, head_dim)      # the cache stays in 16-bit
    for bits in (16, 4):
        w = weights_gb(n, bits)
        verdict = "fits" if w + cache < 3.5 else "too big"     # keep ~0.5 GB for the CUDA context and activations
        print(f"{name} @ {bits:2}-bit: weights {w:5.2f} GB + 8k-token cache {cache:4.2f} GB -> {verdict} in 4 GB")

<div dir="rtl">

### نموذج محادثة حقيقي

</div>

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "Qwen/Qwen2.5-0.5B-Instruct"          # a small multilingual chat model; look for newer versions too
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, dtype="auto", device_map="auto")

messages = [
    {"role": "system", "content": "أنت مساعد مفيد. أجب بالعربية باختصار."},
    {"role": "user", "content": "اشرح الانتباه الذاتي في جملتين."},
]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                       return_tensors="pt", return_dict=True).to(model.device)
out = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7, top_p=0.9)
new_tokens = out[0, inputs["input_ids"].shape[1]:]          # drop the prompt
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

<div dir="rtl">

### نموذج أكبر بأربع بتات

</div>

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16    # older cards: float16
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dtype)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct",
                                             quantization_config=bnb, device_map="auto")
print(f"{model.get_memory_footprint() / 1e9:.2f} GB")

<div dir="rtl">

## تمرين: ذاكرة السياق لنموذجك

نموذجك من الدرس 49: 4 طبقات، و 4 رؤوس، وطول كل رأس 32، وسياق 64 رمزاً. انتباه عادي، أي كل رأس له مفتاح وقيمة.

1. احسب ذاكرة المفاتيح والقيم لسياق كامل بدقة 32 بت، أي 4 بايتات.
2. قارنها بذاكرة الأوزان بنفس الدقة.
3. تحقق بدالة الحاسبة.

</div>

In [ ]:
# اكتب حلّك هنا
